<a href="https://colab.research.google.com/github/miftahulhdd/BigData26_B_2411533005_MiftahulHudaAmri/blob/main/praktikum05/BD_B_P05_2411533005_MiftahulHudaAmrii.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install library

In [1]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

Unduh Data dan Jalankan Portal Lokal

In [2]:
import collections
import http.server
import json
import os
import queue
import re
import threading
import time
from pathlib import Path
import urllib.parse
import urllib.request
import pandas as pd

URL_DATA = (
    "https://raw.githubusercontent.com/ibamibrahim/"
    "dataset-judul-berita-indonesia/master/detik_news_title.csv"
)

if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print(
    "Ukuran berkas :",
    round(os.path.getsize("detik_news_title.csv") / 1e6, 1),
    "MB",
)
print("Baris, kolom :", df_sumber.shape)
print("Kolom :", list(df_sumber.columns))

import html

HALAMAN_UKURAN = 100  # judul per halaman


def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):

        def log_message(self, *args):  # matikan log
            pass

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(
                    200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain"
                )
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[
                    (hal - 1) * HALAMAN_UKURAN : hal * HALAMAN_UKURAN
                ]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime(
                        "%Y-%m-%d"
                    )
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        "</article>"
                    )
                self._kirim(
                    200,
                    f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                    f'<nav data-total="{n_hal}"></nav></body></html>',
                )
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal


server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Ukuran berkas : 16.1 MB
Baris, kolom : (91017, 4)
Kolom : ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


Crawling Portal Lokal

In [3]:
import urllib.robotparser
from bs4 import BeautifulSoup
import requests

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"


def baca_robots(base):
    r = requests.get(
        base + "/robots.txt", headers={"User-Agent": UA}, timeout=10
    )
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp


robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


In [4]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append(
            {
                "judul": link.get_text(strip=True),
                "url": link["href"],
                "kategori": a.select_one("span.kategori").get_text(strip=True),
                "tanggal": a.select_one("time")["datetime"],
            }
        )
    return hasil


def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)


contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


 Crawling Berita Sungguhan dari Sitemap detik.com

In [5]:
import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup

# ID artikel dapat berada di akhir URL atau diikuti path/query.
POLA_ARTIKEL = re.compile(
    r"^https?://([\w-]+)\.detik\.com/(?:[^?#]*/)?d-(\d+)(?:[/?#]|$)"
)


def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict artikel."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []

    for u in sup.find_all("url"):
        loc = u.find("loc")
        judul = u.find("title")
        waktu = u.find("publication_date")

        if loc is None or judul is None:
            continue

        url = loc.get_text(strip=True)
        parsed_url = urlparse(url)
        host = (parsed_url.hostname or "").lower()

        # Kategori diambil dari subdomain, sehingga URL inet tetap dikenali
        # meskipun bentuk path-nya tidak cocok dengan pola ID artikel.
        subdomain = host.split(".")[0] if host.endswith(".detik.com") else None
        m = POLA_ARTIKEL.match(url)
        kategori = subdomain if subdomain and subdomain != "www" else None

        # ID tetap diambil bila pola ID artikel ditemukan.
        id_artikel = int(m.group(2)) if m else None
        kunci = u.find("keywords")
        teks_tanggal = waktu.get_text(strip=True) if waktu else None

        hasil.append(
            {
                "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
                "url": url,
                "kategori": kategori,
                "id": id_artikel,
                "tanggal": teks_tanggal[:10] if teks_tanggal else None,
                "kata_kunci": kunci.get_text(strip=True) if kunci else "",
            }
        )

    return hasil


In [6]:
# Contoh XML
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
 xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
 <url>
 <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisadiberhentikan-tanpa-usulan-dprd]]></loc>
 <news:news>

<news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
 <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
 <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan
DPRD]]></news:title>
 <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
 </news:news>
 </url>
 <url>
<loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwarilewat-posyandu-ternak-keliling]]></loc>
 <news:news>
 <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
 <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak
Keliling]]></news:title>
 </news:news>
 </url>
 <url>
 <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibatbanjir-jadi-sumber-pendapatan-padang]]></loc>
 <news:news>
 <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
 <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan
Padang]]></news:title>
 </news:news>
 </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})

print(
    "kata kunci entri pertama:",
    parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"],
)
print(
    "Sama bila masukan berupa bytes:",
    parse_sitemap_news(CONTOH_XML.encode("utf-8"))
    == parse_sitemap_news(CONTOH_XML),
)

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


In [7]:
import urllib.robotparser

UA_DETIK = (
    "BigData/1.0 (praktikum pembelajaran; kontak: miftahulhudaamrii@gmail.com)"
)
DELAY = 2.0  # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7  # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20  # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]
_robots = {}


def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)


def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(
            f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju."
        )
    r.raise_for_status()
    time.sleep(delay)
    return r.content  # bytes: pengkodean dibaca dari deklarasi XML


def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    url_terlihat = set()

    for sm in SITEMAP_DIPILIH:
        sisa_kuota = min(BATAS_PER_BAGIAN, BATAS_TOTAL - len(hasil))
        if sisa_kuota <= 0:
            break

        daftar = parse_sitemap_news(ambil_xml(sm, sesi))
        host_sitemap = (urlparse(sm).hostname or "").lower()
        kategori_sitemap = host_sitemap.split(".")[0]
        terpilih = 0

        # Lewati URL yang sudah pernah muncul, lalu ambil pengganti dari
        # bagian sitemap yang sama sampai kuota artikel unik terpenuhi.
        for item in daftar:
            url = item.get("url")
            if not url or url in url_terlihat:
                continue

            if not item.get("kategori"):
                item["kategori"] = kategori_sitemap

            url_terlihat.add(url)
            hasil.append(item)
            terpilih += 1

            if terpilih >= sisa_kuota:
                break

    return hasil[:BATAS_TOTAL]


In [8]:
import json
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")

    cek = pd.DataFrame(sampel)
    duplikat = cek["url"].duplicated(keep=False)
    tanpa_kategori = cek[cek["kategori"].isna()]

    print("Judul terkumpul :", len(cek))
    print("Judul kosong    :", (cek["judul"].fillna("").str.strip() == "").sum())
    print("URL duplikat    :", cek["url"].duplicated().sum())
    print("Tanggal kosong  :", cek["tanggal"].isna().sum())
    print("Kategori        :", cek["kategori"].value_counts(dropna=False).to_dict())

    if not tanpa_kategori.empty:
        print("\nURL tanpa kategori (perlu diperiksa):")
        print(tanpa_kategori[["url", "judul"]].to_string(index=False))

    if duplikat.any():
        print("\nURL duplikat (perlu diperiksa):")
        print(
            cek.loc[duplikat, ["url", "kategori", "judul"]]
            .sort_values("url")
            .to_string(index=False)
        )

    print("\nContoh artikel:")
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))


Judul terkumpul : 20
Judul kosong    : 0
URL duplikat    : 0
Tanggal kosong  : 0
Kategori        : {'news': 7, 'inet': 7, 'sport': 6}

Contoh artikel:
                                                                judul kategori    tanggal
Mahasiswa Dibegal Komplotan Bermobil Pikap di Depok, Pelaku Ditangkap     news 2026-11-12
               Komentar Bobby Nasution Usai Gibran Mundur dari Wapres     news 2026-11-12
   Ketua MPR Temui Prabowo Cocokkan Tanggal Sidang Penggantian Wapres     news 2026-11-12


Channel ala Flume ke lapisan_mentah

In [9]:
RAW = Path("lapisan_mentah")
RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}


def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)  # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(
            statistik["max_antrean"], channel.qsize()
        )
    channel.put(STOP)


def sink(ukuran_batch=5000):
    batch, nomor = [], 0

    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []

    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev)
        statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()


t0 = time.perf_counter()
t_src = threading.Thread(target=source)
t_snk = threading.Thread(target=sink)
t_src.start()
t_snk.start()
t_src.join()
t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink :", f"{statistik['keluar']:,}")
print(
    "File batch :",
    statistik["file"],
    "->",
    sorted(p.name for p in RAW.glob("*.jsonl"))[:3],
    "...",
)
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print(
    "Total ukuran mentah :",
    round(sum(p.stat().st_size for p in RAW.glob("*.jsonl")) / 1e6, 1),
    "MB",
)
print("Waktu crawl total :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink : 91,017
File batch : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total : 37.3 detik untuk 911 halaman


Quality, Cleaning, dan Parquet Berpartisi

In [10]:
from pathlib import Path
import pandas as pd

RAW = Path("lapisan_mentah")
berkas_mentah = sorted(RAW.glob("*.jsonl"))

if not berkas_mentah:
    raise FileNotFoundError(
        "Tidak ditemukan file JSONL di folder lapisan_mentah. "
        "Jalankan terlebih dahulu sel Channel ala Flume ke lapisan_mentah."
    )

mentah = pd.concat(
    [pd.read_json(p, lines=True) for p in berkas_mentah],
    ignore_index=True,
)

print("Baris mentah:", f"{len(mentah):,}")
print("Jumlah file:", len(berkas_mentah))


Baris mentah: 91,017
Jumlah file: 19


In [11]:
import pandas as pd

# Pastikan POLA_ARTIKEL sudah di-define sebelumnya, atau gunakan regex langsung jika belum
# POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

KATEGORI_DIKENAL = {
    "finance",
    "food",
    "health",
    "hot",
    "inet",
    "news",
    "oto",
    "sport",
    "travel",
}


def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong": d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata": jml_kata.between(3, 30),
        "kategori dikenal": d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid": pd.to_datetime(
            d["tanggal"], errors="coerce"
        ).notna(),
        "url artikel valid": d["url"].fillna("").str.match(POLA_ARTIKEL),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False),
    }


for nama, lulus in aturan_kualitas(mentah).items():
    print(
        f"{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)"
    )

judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)


In [12]:
bersih = mentah.copy()
bersih["judul"] = (
    bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
)
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = (
    bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")
)  # kosong bila url rusak
bersih["url"] = bersih["url"].where(
    bersih["url"].str.contains(r"/d-\d+", regex=True)
)
print(
    "Baris setelah dibersihkan:",
    f"{len(bersih):,}",
    "| dibuang:",
    f"{len(mentah) - len(bersih):,}",
)

Baris setelah dibersihkan: 90,818 | dibuang: 199


In [13]:
import pyarrow.dataset as ds
import shutil


def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = (
        sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    )
    return n, ukuran


# BAGIAN KUNCI: bandingkan dua pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")

shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print(
    "\nDisimpan ke",
    TERKURASI,
    "dengan partisi kategori:",
    n,
    "file,",
    round(mb, 1),
    "MB",
)

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(
    dataset.get_fragments(filter=ds.field("kategori") == "finance")
)
print(
    "Partition pruning: baca",
    len(hanya),
    "dari",
    len(semua),
    "file untuk kategori = finance",
)

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


Query DuckDB dan Bentuk Dokumen

In [14]:
import duckdb

con = duckdb.connect()
con.sql(
    f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet',"
    " hive_partitioning=true)"
)
print(
    con.sql(
        "SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori"
        " ORDER BY jumlah DESC"
    )
    .df()
    .to_string(index=False)
)
print()
print(
    con.sql("""
 SELECT bulan, COUNT(*) AS judul_corona
 FROM berita WHERE lower(judul) LIKE '%corona%'
 GROUP BY bulan ORDER BY bulan""").df().to_string(index=False)
)

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


Mini MapReduce: Kasus Kantin

In [15]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],  # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],  # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"],  # kasir C
]


def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))


def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))


# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)

print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))


# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())


setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


Word Count Terdistribusi: Partisi sebagai Input Split

In [16]:
STOPWORD = set(
    """yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split()
)


def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())


# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f" {p.parent.name:<16} {n:>6,} judul")


def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]


hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)

hasil_semua = {k: sum(v) for k, v in shuffle.items()}
print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")

print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f" {k:<10} {n:>7,}")

top = [
    (k, n)
    for k, n in collections.Counter(hasil_semua).most_common()
    if k not in STOPWORD
]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f" {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(
    kata for t in bersih["judul"] for kata in tokenisasi(t)
)
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
 kategori=finance 14,141 judul
 kategori=food     4,314 judul
 kategori=health   4,914 judul
 kategori=hot     16,290 judul
 kategori=inet     5,577 judul
 kategori=news    32,321 judul
 kategori=oto      4,375 judul
 kategori=sport    2,433 judul
 kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
 di          27,390
 corona      17,239
 ini         10,124
 yang         8,668
 dan          6,785
 ke           5,592
 tak          5,096
 dari         4,697
 jadi         4,486
 untuk        3,723

10 kata teratas (tanpa stopword):
 corona      17,239
 rp           3,154
 virus        3,075
 kasus        2,743
 polisi       2,732
 positif      2,682
 indonesia    2,504
 warga        2,495
 rumah        2,465
 jokowi       2,317

Sama dengan Counter biasa: True


Combiner dan Jumlah Data yang Berpindah

In [17]:
from multiprocessing import Pool


def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c


def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0


print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")

for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses)
        waktu.append(w)
    print(
        f"{proses} proses, tanpa combiner".ljust(28),
        f"{sorted(waktu)[1]:>10.3f}",
        f"{sum(len(x) for x in h):>20,}",
    )

    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses)
        waktu.append(w)
    print(
        f"{proses} proses, dengan combiner".ljust(28),
        f"{sorted(waktu)[1]:>10.3f}",
        f"{sum(len(x) for x in h2):>20,}",
    )

total = collections.Counter()
for c in h2:
    total.update(c)

print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.318              825,601
1 proses, dengan combiner         0.356               74,009
2 proses, tanpa combiner          1.027              825,601
2 proses, dengan combiner         0.578               74,009

Hasil dengan combiner sama persis: True


Data Skew dan Beban Reducer

In [18]:
import collections
import zlib  # <--- Ditambahkan agar fungsi zlib.crc32 dapat digunakan

R = 4  # jumlah reducer


def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r


def beban(pasangan_per_mapper, r=R):
    b = [0] * r  # Perbaikan sintaks perkalian list [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b


def tampil(nama, b):
    tot = sum(b)
    print(
        f"{nama:<22}",
        " ".join(
            f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)
        ),
    )


tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print(
    "\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):"
)
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(
        f" {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%"
    )

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(
        f" {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di"
        f" reducer R{partitioner(kata)}"
    )

tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
   4 reducer: terberat 29.17% | bagian adil 25.00%
  16 reducer: terberat  8.82% | bagian adil  6.25%
  64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
 di        27,390 pasangan =  3.3% dari total, di reducer R0
 corona    17,239 pasangan =  2.1% dari total, di reducer R2
 ini       10,124 pasangan =  1.2% dari total, di reducer R0
 yang       8,668 pasangan =  1.0% dari total, di reducer R3
 dan        6,785 pasangan =  0.8% dari total, di reducer R3


Hadoop Streaming

In [19]:
import py_compile
from pathlib import Path

Path("masukan.tsv").write_text(
    "\n".join(
        f"{k}\t{j}"
        for k, j in zip(bersih["kategori"], bersih["judul"])
    ) + "\n",
    encoding="utf-8",
)

MAPPER = r'''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\n").split("\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\t1")
'''

REDUCER = r'''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\n").split("\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\t{total}")
'''

Path("mapper.py").write_text(MAPPER, encoding="utf-8")
Path("reducer.py").write_text(REDUCER, encoding="utf-8")

py_compile.compile("mapper.py", doraise=True)
py_compile.compile("reducer.py", doraise=True)

print(
    "Baris masukan:",
    sum(1 for _ in open("masukan.tsv", encoding="utf-8")),
)


Baris masukan: 90818


In [20]:
%%bash
head -n 3 masukan.tsv | python3 mapper.py
time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
wc -l hasil_pipe.txt
sort -k2,2nr hasil_pipe.txt | head -n 5


kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785



real	0m1.162s
user	0m1.249s
sys	0m0.057s


In [21]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)

print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman

In [22]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c


gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))

for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3

SIAP = Path("siap_pakai")
SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print(
    "Baris diekspor:",
    db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0],
)
print(
    db.execute(
        """SELECT kata, jumlah FROM kata_per_kategori
 WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3"""
    ).fetchall()
)
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


Latihan 1: Stopword domain

In [23]:
# ===== Latihan 1: Stopword domain =====
TAMBAHAN = {"rp", "ri", "the", "foto", "galeri"}   # 5 kata tambahan
STOPWORD_BARU = STOPWORD | TAMBAHAN                # STOPWORD asli tidak diubah


def hitung_per_kategori(stopword):
    """Mapper (kategori, kata) + combiner untuk semua split, dengan stopword tertentu."""
    total = collections.Counter()
    for path in splits:
        kategori = path.parent.name.split("=")[1]
        judul = pd.read_parquet(path, columns=["judul"])["judul"]
        for t in judul:
            for kata in tokenisasi(t):
                if kata not in stopword:
                    total[(kategori, kata)] += 1
    return total


def top5(total):
    per_kat = collections.defaultdict(list)
    for (kat, kata), n in total.items():
        per_kat[kat].append((n, kata))
    return {kat: sorted(v, reverse=True)[:5] for kat, v in per_kat.items()}


lama = top5(hitung_per_kategori(STOPWORD))
baru = top5(hitung_per_kategori(STOPWORD_BARU))
if "gabung" in globals():
    print("Hasil 'lama' cocok dengan H-13:", lama == top5(gabung))

print("Kata yang ditambahkan:", sorted(TAMBAHAN), "\n")
for kat in sorted(baru):
    k_lama = [k for _, k in lama[kat]]
    k_baru = [k for _, k in baru[kat]]
    print(f"{kat:<8} SEBELUM: {', '.join(f'{k} ({n:,})' for n, k in lama[kat])}")
    print(f"{'':<8} SESUDAH: {', '.join(f'{k} ({n:,})' for n, k in baru[kat])}")
    keluar, masuk = [k for k in k_lama if k not in k_baru], [k for k in k_baru if k not in k_lama]
    print(f"{'':<8} keluar: {keluar or '-'} | masuk: {masuk or '-'}\n")

Hasil 'lama' cocok dengan H-13: True
Kata yang ditambahkan: ['foto', 'galeri', 'ri', 'rp', 'the'] 

finance  SEBELUM: corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
         SESUDAH: corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)
         keluar: ['rp', 'ri'] | masuk: ['mau', 'ekonomi']

food     SEBELUM: makanan (699), makan (539), foto (458), galeri (438), enak (358)
         SESUDAH: makanan (699), makan (539), enak (358), bikin (343), rumah (311)
         keluar: ['foto', 'galeri'] | masuk: ['bikin', 'rumah']

health   SEBELUM: corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
         SESUDAH: corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
         keluar: - | masuk: -

hot      SEBELUM: corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
         SESUDAH: corona (1,658), anak (793), rumah (561), meninggal (545), film (489)
         keluar: ['the'] | masuk: ['film']



Latihan 2: Key (bulan, kata)

In [24]:
# ===== Latihan 2: key (bulan, kata) =====
SW = STOPWORD_BARU     # dari Latihan 1; ganti ke STOPWORD bila ingin versi asli


def mapper_bulan(path):
    """Key gabungan (bulan, kata); combiner di dalam mapper."""
    d = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for judul, bulan in zip(d["judul"], d["bulan"]):
        for kata in tokenisasi(judul):
            if kata not in SW:
                c[(bulan, kata)] += 1
    return c


gabung_bulan = collections.Counter()
for c in map(mapper_bulan, splits):
    gabung_bulan.update(c)


def kata_bulan(bulan):
    return {k: n for (b, k), n in gabung_bulan.items() if b == bulan}


jan, mei = kata_bulan("2020-01"), kata_bulan("2020-05")
for nama, d in (("Januari 2020", jan), ("Mei 2020", mei)):
    print(f"10 kata teratas {nama}:")
    print("  " + ", ".join(f"{k} ({n:,})" for k, n in collections.Counter(d).most_common(10)))

# Perubahan porsi kata (dinormalisasi, karena jumlah total kata tiap bulan berbeda)
tot_jan, tot_mei = sum(jan.values()), sum(mei.values())
kandidat = [k for k in set(jan) | set(mei) if jan.get(k, 0) + mei.get(k, 0) >= 50]


def rasio(k):   # >1 berarti porsinya naik dari Januari ke Mei
    return ((mei.get(k, 0) + 1) / tot_mei) / ((jan.get(k, 0) + 1) / tot_jan)


print("\nKata yang porsinya NAIK paling besar (Jan -> Mei):")
for k in sorted(kandidat, key=rasio, reverse=True)[:10]:
    print(f"  {k:<14} Jan={jan.get(k, 0):>5,}  Mei={mei.get(k, 0):>5,}  x{rasio(k):.1f}")
print("\nKata yang porsinya TURUN paling besar (Jan -> Mei):")
for k in sorted(kandidat, key=rasio)[:10]:
    print(f"  {k:<14} Jan={jan.get(k, 0):>5,}  Mei={mei.get(k, 0):>5,}  x{rasio(k):.2f}")

10 kata teratas Januari 2020:
  banjir (1,220), polisi (554), jokowi (516), tahun (493), baru (475), mobil (428), minta (427), kpk (415), indonesia (410), china (386)
10 kata teratas Mei 2020:
  corona (2,935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562), covid (522), psbb (522), polisi (517), normal (504)

Kata yang porsinya NAIK paling besar (Jan -> Mei):
  pandemi        Jan=    0  Mei=  596  x577.1
  psbb           Jan=    0  Mei=  522  x505.5
  covid          Jan=    0  Mei=  522  x505.5
  thr            Jan=    0  Mei=  171  x166.3
  idul           Jan=    0  Mei=  125  x121.8
  protokol       Jan=    0  Mei=  106  x103.4
  sahur          Jan=    0  Mei=  100  x97.6
  mei            Jan=    4  Mei=  416  x80.6
  gugus          Jan=    0  Mei=   75  x73.5
  kempot         Jan=    2  Mei=  194  x62.8

Kata yang porsinya TURUN paling besar (Jan -> Mei):
  imlek          Jan=  161  Mei=    0  x0.01
  masters        Jan=  126  Mei=    0  x0.01
  asabri       

Latihan 3: Bigram

In [25]:
# ===== Latihan 3: Bigram =====
def mapper_bigram(path):
    """Kembalikan (jumlah pasangan tanpa combiner, Counter bigram = dengan combiner)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    n_pasangan, c = 0, collections.Counter()
    for t in judul:
        tok = tokenisasi(t)
        n_pasangan += max(len(tok) - 1, 0)
        c.update(f"{a} {b}" for a, b in zip(tok, tok[1:]))
    return n_pasangan, c


hasil_bigram = [mapper_bigram(p) for p in splits]
tanpa_combiner = sum(n for n, _ in hasil_bigram)
dengan_combiner = sum(len(c) for _, c in hasil_bigram)

total_bigram = collections.Counter()
for _, c in hasil_bigram:
    total_bigram.update(c)

n_token = sum(len(tokenisasi(t)) for t in bersih["judul"])
n_judul_bertoken = sum(1 for t in bersih["judul"] if tokenisasi(t))

print(f"Pasangan bigram tanpa combiner : {tanpa_combiner:,}")
print(f"Pasangan bigram dengan combiner: {dengan_combiner:,}  (turun {tanpa_combiner/dengan_combiner:.1f}x)")
print(f"Bigram unik (key setelah shuffle): {len(total_bigram):,}")
print("Cek: token - judul bertoken =", f"{n_token - n_judul_bertoken:,}",
      "| sama?", tanpa_combiner == n_token - n_judul_bertoken)
print("Cek: jumlah hitungan bigram = pasangan tanpa combiner?", sum(total_bigram.values()) == tanpa_combiner)
print("\n'virus corona':", f"{total_bigram['virus corona']:,}")

print("\n10 bigram teratas (semua):")
for k, n in total_bigram.most_common(10):
    print(f"  {k:<22} {n:>7,}")
print("\n10 bigram teratas (tanpa stopword di kedua kata):")
bersih_sw = [(k, n) for k, n in total_bigram.most_common()
             if not any(w in STOPWORD for w in k.split())]
for k, n in bersih_sw[:10]:
    print(f"  {k:<22} {n:>7,}")

Pasangan bigram tanpa combiner : 734,783
Pasangan bigram dengan combiner: 411,217  (turun 1.8x)
Bigram unik (key setelah shuffle): 349,114
Cek: token - judul bertoken = 734,783 | sama? True
Cek: jumlah hitungan bigram = pasangan tanpa combiner? True

'virus corona': 2,915

10 bigram teratas (semua):
  virus corona             2,915
  corona di                2,178
  positif corona           1,804
  di tengah                1,258
  pandemi corona           1,075
  di rumah                 1,047
  new normal                 917
  di indonesia               871
  rp juta                    700
  hari ini                   627

10 bigram teratas (tanpa stopword di kedua kata):
  virus corona             2,915
  positif corona           1,804
  pandemi corona           1,075
  new normal                 917
  rp juta                    700
  rp m                       606
  pasien corona              600
  cegah corona               583
  rp t                       549
  tengah pandemi     

Latihan 4: Jumlah reducer

In [26]:
# ===== Latihan 4: Jumlah reducer =====
pasangan_kata = [mapper_split(p) for p in splits]          # tanpa combiner
counter_kata = [mapper_combiner(p) for p in splits]        # dengan combiner


def beban_tanpa(r):
    b = [0] * r
    for h in pasangan_kata:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b


def beban_dengan(r):
    b = [0] * r
    for c in counter_kata:
        for kata in c:
            b[partitioner(kata, r)] += 1
    return b


print(f"{'R':>3} | {'adil':>6} | {'terberat':>8} {'rasio':>6} (tanpa combiner) | "
      f"{'terberat':>8} {'rasio':>6} (dengan combiner)")
for r in (2, 4, 8, 16, 32, 64):          # 2, 8, 32 = permintaan latihan; 4, 16, 64 = pembanding H-11
    adil = 100 / r
    t = max(beban_tanpa(r)) / sum(beban_tanpa(r)) * 100
    d = max(beban_dengan(r)) / sum(beban_dengan(r)) * 100
    tanda = "" if r in (2, 8, 32) else "  <- pembanding H-11"
    print(f"{r:>3} | {adil:5.2f}% | {t:7.2f}% {t/adil:5.2f}x                    | "
          f"{d:7.2f}% {d/adil:5.2f}x{tanda}")

  R |   adil | terberat  rasio (tanpa combiner) | terberat  rasio (dengan combiner)
  2 | 50.00% |   52.27%  1.05x                    |   50.23%  1.00x
  4 | 25.00% |   29.17%  1.17x                    |   25.55%  1.02x  <- pembanding H-11
  8 | 12.50% |   16.83%  1.35x                    |   12.85%  1.03x
 16 |  6.25% |    8.82%  1.41x                    |    6.54%  1.05x  <- pembanding H-11
 32 |  3.12% |    5.82%  1.86x                    |    3.44%  1.10x
 64 |  1.56% |    4.28%  2.74x                    |    1.86%  1.19x  <- pembanding H-11


Latihan 5: Crawl ulang dengan jeda

In [27]:
# ===== Latihan 5: Crawl ulang dengan jeda =====
def ukur_crawl(n_halaman, delay):
    t0 = time.perf_counter()
    n = sum(1 for _ in crawl_iter(BASE, range(1, n_halaman + 1), delay=delay))
    return n, time.perf_counter() - t0


n0, t_tanpa = ukur_crawl(50, 0.0)
n1, t_jeda = ukur_crawl(50, 0.05)
print(f"50 halaman, jeda 0 detik   : {t_tanpa:6.2f} detik ({n0} judul)")
print(f"50 halaman, jeda 0,05 detik: {t_jeda:6.2f} detik ({n1} judul)")
print(f"Selisih: {t_jeda - t_tanpa:.2f} detik (teori: 50 x 0,05 = 2,5 detik)")

# Perkiraan 911 halaman dengan jeda 1 detik (HANYA DIHITUNG, jangan dijalankan)
WAKTU_DASAR_911 = 50.8        # detik, hasil H-5 pada notebookmu (jeda 0)
perkiraan = WAKTU_DASAR_911 + 911 * 1.0
print(f"\nPerkiraan 911 halaman, jeda 1 detik: {WAKTU_DASAR_911} + 911 x 1 = "
      f"{perkiraan:,.1f} detik = {perkiraan/60:.1f} menit")

50 halaman, jeda 0 detik   :   3.66 detik (5000 judul)
50 halaman, jeda 0,05 detik:   5.22 detik (5000 judul)
Selisih: 1.57 detik (teori: 50 x 0,05 = 2,5 detik)

Perkiraan 911 halaman, jeda 1 detik: 50.8 + 911 x 1 = 961.8 detik = 16.0 menit


Latihan 6: Sitemap lain

In [28]:
# ===== Latihan 6 (bagian A): =====
cek20 = pd.read_json("contoh_crawl/detik_sitemap.jsonl", lines=True)
cek20["tanggal"] = cek20["tanggal"].astype("string").str[:10]   # ambil tanggal saja, sesuai catatan zona waktu modul
print("Jumlah baris:", len(cek20))
print("Kategori kosong (tidak cocok POLA_ARTIKEL):", cek20["kategori"].isna().sum())
print("URL duplikat:", cek20["url"].duplicated().sum())
print("\nBaris bermasalah (kategori kosong):")
print(cek20.loc[cek20["kategori"].isna(), ["judul", "url"]].to_string(index=False, max_colwidth=70))

print("\nHasil aturan_kualitas (H-6) pada hasil crawl:")
for nama, lulus in aturan_kualitas(cek20).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>3} dari {len(cek20)}")

Jumlah baris: 20
Kategori kosong (tidak cocok POLA_ARTIKEL): 0
URL duplikat: 0

Baris bermasalah (kategori kosong):
Empty DataFrame
Columns: [judul, url]
Index: []

Hasil aturan_kualitas (H-6) pada hasil crawl:
judul tidak kosong       gagal:   0 dari 20
judul 3-30 kata          gagal:   0 dari 20
kategori dikenal         gagal:   0 dari 20
tanggal valid            gagal:   0 dari 20
url artikel valid        gagal:   0 dari 20
judul tidak duplikat     gagal:   0 dari 20


In [29]:
# ===== Latihan 6 (bagian B): ganti satu sitemap.  =====
INDEKS_SITEMAP = "https://www.detik.com/sitemap.xml"   # periksa dulu di browser; ganti bila alamatnya berbeda
SITEMAP_BARU = None                                    # atau isi manual, mis. "https://.../sitemap_news.xml"

sesi = requests.Session()
sesi.headers["User-Agent"] = UA_DETIK
try:
    if SITEMAP_BARU is None:
        sup = BeautifulSoup(ambil_xml(INDEKS_SITEMAP, sesi), "xml")
        semua = [l.get_text(strip=True) for l in sup.find_all("loc")]
        kandidat = [u for u in semua
                    if u.endswith("sitemap_news.xml") and u not in SITEMAP_DIPILIH]
        print(f"Indeks berisi {len(semua)} entri; {len(kandidat)} sitemap_news lain. Contoh:")
        for u in kandidat[:8]:
            print("  ", u)
        SITEMAP_BARU = kandidat[0]
    print("\nSitemap baru:", SITEMAP_BARU)
    baru = parse_sitemap_news(ambil_xml(SITEMAP_BARU, sesi))[:6]

    lama14 = pd.read_json("contoh_crawl/detik_sitemap.jsonl", lines=True).head(14).to_dict("records")
    gabungan = pd.DataFrame(lama14 + baru)
    gabungan["tanggal"] = gabungan["tanggal"].astype("string").str[:10]
    print("Kategori hasil gabungan:", gabungan["kategori"].value_counts(dropna=False).to_dict())
    for nama, lulus in aturan_kualitas(gabungan).items():
        print(f"{nama:<24} gagal: {(~lulus).sum():>3} dari {len(gabungan)}")
except Exception as e:
    print("Gagal:", type(e).__name__, "-", e)
    print("Isi SITEMAP_BARU secara manual dari sitemap.xml, lalu jalankan sel ini sekali lagi.")

Indeks berisi 536 entri; 245 sitemap_news lain. Contoh:
   https://news.detik.com/suara-pembaca/sitemap_news.xml
   https://news.detik.com/internasional/sitemap_news.xml
   https://news.detik.com/bbc-world/sitemap_news.xml
   https://news.detik.com/abc-australia/sitemap_news.xml
   https://news.detik.com/dw/sitemap_news.xml
   https://news.detik.com/kolom/sitemap_news.xml
   https://news.detik.com/prokontra/sitemap_news.xml
   https://news.detik.com/pemilu/sitemap_news.xml

Sitemap baru: https://news.detik.com/suara-pembaca/sitemap_news.xml
Kategori hasil gabungan: {'news': 13, 'inet': 7}
judul tidak kosong       gagal:   0 dari 20
judul 3-30 kata          gagal:   0 dari 20
kategori dikenal         gagal:   0 dari 20
tanggal valid            gagal:   0 dari 20
url artikel valid        gagal:   0 dari 20
judul tidak duplikat     gagal:   0 dari 20


Latihan 7: Kata kunci

In [30]:
# ===== Latihan 7: kata kunci terbanyak pada hasil crawl =====
def mapper_kunci(baris_json):
    """Satu baris JSONL -> pasangan (kata_kunci, 1)."""
    rekaman = json.loads(baris_json)
    for k in (rekaman.get("kata_kunci") or "").split(","):
        k = k.strip().lower()
        if k:
            yield (k, 1)


def reducer_kunci(kunci, daftar_nilai):
    return (kunci, sum(daftar_nilai))


with open("contoh_crawl/detik_sitemap.jsonl", encoding="utf-8") as f:
    pasangan = [p for baris in f for p in mapper_kunci(baris)]

shuffle_kunci = collections.defaultdict(list)          # tahap shuffle
for k, n in pasangan:
    shuffle_kunci[k].append(n)
hasil_kunci = dict(reducer_kunci(k, v) for k, v in shuffle_kunci.items())

print("Pasangan dari mapper:", len(pasangan), "| kata kunci unik:", len(hasil_kunci))
if hasil_kunci:
    for k, n in sorted(hasil_kunci.items(), key=lambda x: (-x[1], x[0]))[:10]:
        print(f"  {k:<30} {n}")
else:
    print("Tidak ada kata kunci pada hasil crawl (kolom kata_kunci kosong untuk semua baris).")

Pasangan dari mapper: 103 | kata kunci unik: 86
  piala dunia 2026               6
  gibran rakabuming raka         3
  jabodetabek                    3
  jensen huang                   3
  ceo nvidia                     2
  final piala dunia 2026         2
  nvidia                         2
  timnas spanyol                 2
  uefa nations league            2
  wakil presiden                 2


tugas mandiri

In [31]:
# ===== Tugas Mandiri: Inverted Index dengan MapReduce =====
OFFSET = 1_000_000          # batas jumlah baris per split (untuk id sintetis)
BUANG_STOPWORD = False      # True bila ingin indeks tanpa stopword


def mapper_inverted(arg):
    """Satu mapper = satu split. Keluaran: daftar pasangan (kata, id_judul)."""
    i, path = arg
    d = pd.read_parquet(path, columns=["id", "judul"])
    pasangan = []
    for k, (id_, judul) in enumerate(zip(d["id"], d["judul"])):
        # id kosong (kategori inet) -> id sintetis NEGATIF, unik per split & baris
        id_ = -(i * OFFSET + k + 1) if pd.isna(id_) else int(id_)
        for kata in set(tokenisasi(judul)):              # set: satu kata sekali per judul
            if BUANG_STOPWORD and kata in STOPWORD:
                continue
            pasangan.append((kata, id_))
    return pasangan


def reducer_inverted(kata, daftar_id):
    """Gabungkan id menjadi daftar unik; id asli (positif) lebih dulu, lalu id sintetis."""
    return kata, sorted(set(daftar_id), key=lambda x: (x < 0, abs(x)))


# MAP
hasil_map_inv = [mapper_inverted(a) for a in enumerate(splits)]
n_pasangan_inv = sum(len(h) for h in hasil_map_inv)

# SHUFFLE: partitioner CRC32 (sama seperti H-11) ke R reducer
R_INV = 4
partisi = [collections.defaultdict(list) for _ in range(R_INV)]
for h in hasil_map_inv:
    for kata, id_ in h:
        partisi[partitioner(kata, R_INV)][kata].append(id_)

# REDUCE
indeks = {}
for r in range(R_INV):
    for kata, daftar in partisi[r].items():
        k, ids = reducer_inverted(kata, daftar)
        indeks[k] = ids

print("Pasangan (kata, id) dari mapper:", f"{n_pasangan_inv:,}")
print("Kata unik di indeks            :", f"{len(indeks):,}")
print("Beban reducer (jumlah kata)    :", [len(p) for p in partisi])
n_sintetis = sum(1 for h in hasil_map_inv for _, i_ in h if i_ < 0)
print("Pasangan memakai id sintetis   :", f"{n_sintetis:,}")

# Verifikasi silang: jumlah judul yang memuat kata = panjang daftar id
for kata in ("corona", "jokowi", "wisata"):
    nyata = sum(1 for t in bersih["judul"] if kata in tokenisasi(t))
    print(f"cek '{kata}': indeks={len(indeks[kata]):,} | hitung langsung={nyata:,} | sama? {len(indeks[kata]) == nyata}")

# Tiga kata contoh + lima id pertama (untuk laporan)
judul_by_id = dict(zip(bersih["id"].dropna().astype(int), bersih.loc[bersih["id"].notna(), "judul"]))
print()
for kata in ("corona", "jokowi", "wisata"):
    lima = indeks[kata][:5]
    print(f"{kata}: {len(indeks[kata]):,} judul | 5 id pertama: {lima}")
    for i_ in lima:
        print(f"     {i_}: {judul_by_id.get(i_, '(id sintetis)')}")

# Simpan ke siap_pakai/
SIAP.mkdir(exist_ok=True)
pd.DataFrame({"kata": list(indeks), "df": [len(v) for v in indeks.values()],
              "ids": list(indeks.values())}).to_parquet(SIAP / "inverted_index.parquet", index=False)
print("\nTersimpan:", SIAP / "inverted_index.parquet")

Pasangan (kata, id) dari mapper: 817,993
Kata unik di indeks            : 32,275
Beban reducer (jumlah kata)    : [7942, 8053, 8162, 8118]
Pasangan memakai id sintetis   : 43,634
cek 'corona': indeks=17,228 | hitung langsung=17,228 | sama? True
cek 'jokowi': indeks=2,316 | hitung langsung=2,316 | sama? True
cek 'wisata': indeks=606 | hitung langsung=606 | sama? True

corona: 17,228 judul | 5 id pertama: [4856495, 4856598, 4868317, 4868545, 4869163]
     4856495: WHO Sebut Penyakit Misterius di China Disebabkan oleh Virus Corona
     4856598: Seberapa Berbahaya Virus Corona di Balik 'Penyakit Misterius' di China?
     4868317: Peta Persebaran Virus Corona dari China, Ini Negara-negara yang Terdampak
     4868545: Virus Corona Baru Terkonfirmasi di AS, WHO Adakan Pertemuan Darurat
     4869163: Korut Bentengi Diri dari Virus Corona, Turis Dilarang Masuk!
jokowi: 2,316 judul | 5 id pertama: [4842674, 4842690, 4842703, 4842774, 4843215]
     4842674: Jakarta Banjir, Jokowi: Nomorsatukan Ke